# OneLake Security Error — Detect & Refresh

A self-contained Fabric notebook that **detects OneLake security errors** on your
semantic models and **automatically refreshes** only the affected models to
re-sync the security configuration.

## How it works

1. Queries every configured semantic model concurrently with a lightweight DAX query.
2. If a model returns a OneLake security error, it is flagged for refresh.
3. Models that errored are refreshed concurrently via XMLA.
   - An `automatic` refresh is attempted first (fast, incremental).
   - If that doesn't register, a `full` refresh is sent as a fallback.
4. Models that returned data successfully are left untouched.

## When to use this

OneLake security errors occur when the security configuration on a Direct Lake
semantic model falls out of sync — typically after workspace role changes, item
permission updates, or transient platform issues. A semantic model refresh
re-syncs the configuration and clears the error.

Attach this notebook to a **Fabric schedule** (e.g., every 10–15 minutes) for
automated detection and self-healing, or run it manually when users report
access issues.

## Setup

Edit the **Configuration** cell below with your workspace and model names,
then run all cells.

> **Requirements:** Fabric notebook with a Spark session.  
> **Permissions:** You need Contributor (or higher) on the workspace and
> XMLA read/write enabled on the capacity.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Set your workspace name and the semantic models you want to monitor.

WORKSPACE = "Your-Workspace-Name"  # ← change this

MODELS = [
    "Semantic Model A",  # ← change these to your model names
    "Semantic Model B",
    # Add as many models as you like — they are queried concurrently.
]

# Lightweight DAX query used to probe each model.
# Any valid query works; this just needs to hit the engine.
DAX_QUERY = "EVALUATE ROW(\"OK\", 1)"

In [ ]:
import sempy.fabric as fabric
import json
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from notebookutils import mssparkutils

In [ ]:
# ── OneLake security error detection ──────────────────────────────────────────
# These are the known error messages returned by the AS engine when the
# OneLake security configuration is out of sync on a Direct Lake model.

ONELAKE_ERROR_PATTERNS = [
    "OneLake security configuration has changed",
    "transient issue when trying to determine user permissions defined in OneLake",
    "Universal security version mismatch error on artifact",
]


def _is_onelake_security_error(error_text: str) -> bool:
    lower = error_text.lower()
    return any(p.lower() in lower for p in ONELAKE_ERROR_PATTERNS)


def _probe_model(model_name: str) -> dict:
    """Query a single model. Returns a dict with the result."""
    try:
        fabric.evaluate_dax(
            dataset=model_name,
            workspace=WORKSPACE,
            dax_string=DAX_QUERY,
        )
        return {"model": model_name, "status": "ok"}
    except Exception as e:
        error_text = str(e)
        if _is_onelake_security_error(error_text):
            return {"model": model_name, "status": "security_error", "error": error_text}
        return {"model": model_name, "status": "other_error", "error": error_text}


# Probe all models concurrently
print(f"Probing {len(MODELS)} model(s) in '{WORKSPACE}'...\n")
results = []

with ThreadPoolExecutor(max_workers=len(MODELS)) as executor:
    futures = {executor.submit(_probe_model, m): m for m in MODELS}
    for future in as_completed(futures):
        r = future.result()
        results.append(r)
        icon = "✓" if r["status"] == "ok" else "✗" if r["status"] == "security_error" else "⚠"
        detail = "" if r["status"] == "ok" else f" — {r['error'][:120]}"
        print(f"  {icon} {r['model']}: {r['status']}{detail}")

errors_detected = [r for r in results if r["status"] == "security_error"]
print(f"\nOneLake security errors: {len(errors_detected)} of {len(MODELS)} model(s)")

In [ ]:
# ── Refresh only the affected models ─────────────────────────────────────────
if not errors_detected:
    print("No OneLake security errors detected — nothing to refresh. ✓")
else:
    workspace_id = fabric.resolve_workspace_id(WORKSPACE)

    # Load the .NET TOM assembly for XMLA commands
    _loader = fabric.create_tom_server(readonly=True, workspace=workspace_id)
    _loader.Dispose()
    import Microsoft.AnalysisServices.Tabular as TOM

    def _send_xmla_refresh(dataset_name: str, refresh_type: str):
        """Send a synchronous XMLA refresh command."""
        token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
        conn_str = (
            f"Provider=MSOLAP;"
            f"Data Source=powerbi://api.powerbi.com/v1.0/myorg/{WORKSPACE};"
            f"Password={token};"
        )
        tmsl = json.dumps({"refresh": {"type": refresh_type, "objects": [{"database": dataset_name}]}})
        server = TOM.Server()
        server.Connect(conn_str)
        try:
            result = server.Execute(tmsl)
            for i in range(result.Count):
                for j in range(result[i].Messages.Count):
                    msg = result[i].Messages[j]
                    if hasattr(msg, "Description") and msg.Description:
                        raise RuntimeError(msg.Description)
        finally:
            server.Disconnect()
            server.Dispose()

    def _refresh_model(model_name: str):
        """Attempt automatic refresh; fall back to full if it doesn't register."""
        print(f"\n{'='*60}")
        print(f"Refreshing: {model_name}")
        print(f"{'='*60}")

        # Step 1 — automatic refresh (fast, incremental)
        automatic_ok = True
        start_ms = int(time.time() * 1000)
        print(f"  → Sending automatic refresh...")
        try:
            _send_xmla_refresh(model_name, "automatic")
            print(f"  → Automatic refresh command completed.")
        except RuntimeError as e:
            print(f"  → Automatic refresh error: {e}")
            automatic_ok = False

        # Step 2 — verify it registered; fall back to full if needed
        need_full = not automatic_ok
        if automatic_ok:
            refreshes = fabric.list_refresh_requests(dataset=model_name, workspace=WORKSPACE)
            if refreshes is not None and len(refreshes) > 0:
                latest = refreshes.iloc[0]
                latest_start = (
                    int(latest["Start Time"].timestamp() * 1000)
                    if hasattr(latest["Start Time"], "timestamp")
                    else int(latest["Start Time"])
                )
                if latest_start > start_ms and latest["Status"] == "Completed":
                    print(f"  ✓ Automatic refresh verified (request: {latest['Request Id']})")
                    need_full = False
                else:
                    need_full = True
            else:
                need_full = True

        if need_full:
            print(f"  → Falling back to full refresh...")
            _send_xmla_refresh(model_name, "full")
            print(f"  ✓ Full refresh completed.")

    # Refresh all affected models concurrently
    models_to_refresh = [r["model"] for r in errors_detected]
    failed = []

    print(f"Refreshing {len(models_to_refresh)} affected model(s)...")

    with ThreadPoolExecutor(max_workers=len(models_to_refresh)) as executor:
        futures = {executor.submit(_refresh_model, m): m for m in models_to_refresh}
        for future in as_completed(futures):
            model = futures[future]
            try:
                future.result()
            except Exception as e:
                print(f"\n*** ERROR refreshing {model}: {e} ***")
                failed.append(model)

    print(f"\n{'='*60}")
    print(f"Done. {len(models_to_refresh) - len(failed)}/{len(models_to_refresh)} model(s) refreshed successfully.")
    if failed:
        print(f"FAILED: {failed}")
    print(f"{'='*60}")